# Lakehouse bootstrap v2 — Medallion layout

Run ONCE before deploying. Creates the Delta tables organised in three layers:

**Gold** (analytics-ready)
- `pr_results`        — one row per PR test execution
- `plants`            — plant dimension (code, name, capacity)
- `pending_runs`      — files queued for the Power BI selector

**Silver / Processed** (canonical SCADA + PVSyst, multi-plant, partitioned by plant_code)
- `proc_scada`        — every SCADA row, every plant, canonical column names
- `proc_pvsyst`       — every PVSyst row, every plant, canonical column names

**Catalog**
- `ingested_files`    — which CSVs have been processed (idempotency guard)

In [ ]:
from pathlib import Path
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    IntegerType, LongType, DoubleType, TimestampType
)

LAKEHOUSE_FILES = '/lakehouse/default/Files'
for sub in ['output', 'configs']:
    Path(f'{LAKEHOUSE_FILES}/{sub}').mkdir(parents=True, exist_ok=True)
print('Files/ structure ready (no raw/ — comes from SharePoint shortcut)')

In [ ]:
# Gold: pr_results
pr_results_schema = StructType([
    StructField('run_id',           StringType(),    False),
    StructField('plant_code',       StringType(),    False),
    StructField('plant_name',       StringType(),    True),
    StructField('period_start',     StringType(),    True),
    StructField('period_end',       StringType(),    True),
    StructField('resolution',       StringType(),    True),
    StructField('passed',           BooleanType(),   True),
    StructField('measured_kwh',     DoubleType(),    True),
    StructField('guaranteed_kwh',   DoubleType(),    True),
    StructField('expected_kwh',     DoubleType(),    True),
    StructField('delta_kwh',        DoubleType(),    True),
    StructField('delta_pct',        DoubleType(),    True),
    StructField('valid_days',       IntegerType(),   True),
    StructField('total_intervals',  IntegerType(),   True),
    StructField('valid_intervals',  IntegerType(),   True),
    StructField('c1_failed',        IntegerType(),   True),
    StructField('c2_failed',        IntegerType(),   True),
    StructField('c3_failed',        IntegerType(),   True),
    StructField('c4_failed',        IntegerType(),   True),
    StructField('c5_failed',        IntegerType(),   True),
    StructField('c6_failed',        IntegerType(),   True),
    StructField('aggregated_valid_hours', DoubleType(), True),
    StructField('daily_json',       StringType(),    True),
    StructField('report_html_path', StringType(),    True),
    StructField('report_pdf_path',  StringType(),    True),
    StructField('executed_at',      TimestampType(), True),
])
spark.createDataFrame([], pr_results_schema).write.format('delta').mode('overwrite').saveAsTable('pr_results')
print('pr_results ready')

In [ ]:
# Gold: plants dimension — start with Plaza II only
plants_data = [
    ('PL2', 'Plaza II', 16800, 18310, 0.98),
]
plants_schema = StructType([
    StructField('plant_code',     StringType(),  False),
    StructField('plant_name',     StringType(),  False),
    StructField('ac_capacity_kw', IntegerType(), True),
    StructField('dc_capacity_kw', IntegerType(), True),
    StructField('guaranteed_epi', DoubleType(),  True),
])
spark.createDataFrame(plants_data, plants_schema).write.format('delta').mode('overwrite').saveAsTable('plants')
print(f'plants ready ({len(plants_data)} row)')

In [ ]:
# Gold: pending_runs — populated by the ingest notebook
pending_runs_schema = StructType([
    StructField('plant_code',   StringType(),    False),
    StructField('label',        StringType(),    False),
    StructField('period_start', TimestampType(), True),
    StructField('period_end',   TimestampType(), True),
    StructField('row_count',    LongType(),      True),
    StructField('ingested_at',  TimestampType(), True),
])
spark.createDataFrame([], pending_runs_schema).write.format('delta').mode('overwrite').saveAsTable('pending_runs')
print('pending_runs ready')

In [ ]:
# Silver: proc_scada — partitioned by plant_code
proc_scada_schema = StructType([
    StructField('plant_code',   StringType(),    False),
    StructField('Date',         TimestampType(), False),
    StructField('GHI_01',       DoubleType(),    True),
    StructField('POA_01',       DoubleType(),    True),
    StructField('POA_02',       DoubleType(),    True),
    StructField('POA_03',       DoubleType(),    True),
    StructField('REF_01',       DoubleType(),    True),
    StructField('REF_02',       DoubleType(),    True),
    StructField('WS_01',        DoubleType(),    True),
    StructField('T_AMB_01',     DoubleType(),    True),
    StructField('E_GRID_01',    DoubleType(),    True),
    StructField('SETPOINT_01',  DoubleType(),    True),
    StructField('source_file',  StringType(),    True),
    StructField('ingested_at',  TimestampType(), True),
])
spark.createDataFrame([], proc_scada_schema).write.format('delta').partitionBy('plant_code').mode('overwrite').saveAsTable('proc_scada')
print('proc_scada ready (partitioned by plant_code)')

In [ ]:
# Silver: proc_pvsyst — partitioned by plant_code
proc_pvsyst_schema = StructType([
    StructField('plant_code',  StringType(),    False),
    StructField('Date',        TimestampType(), False),
    StructField('GHI',         DoubleType(),    True),
    StructField('T_AMB',       DoubleType(),    True),
    StructField('ALB',         DoubleType(),    True),
    StructField('POA',         DoubleType(),    True),
    StructField('E_Grid',      DoubleType(),    True),
    StructField('source_file', StringType(),    True),
    StructField('ingested_at', TimestampType(), True),
])
spark.createDataFrame([], proc_pvsyst_schema).write.format('delta').partitionBy('plant_code').mode('overwrite').saveAsTable('proc_pvsyst')
print('proc_pvsyst ready')

In [ ]:
# Catalog: ingested_files — idempotency guard
ingested_files_schema = StructType([
    StructField('plant_code',  StringType(),    False),
    StructField('file_path',   StringType(),    False),
    StructField('file_kind',   StringType(),    False),
    StructField('file_size',   LongType(),      True),
    StructField('row_count',   LongType(),      True),
    StructField('period_start',TimestampType(), True),
    StructField('period_end',  TimestampType(), True),
    StructField('resolution',  StringType(),    True),
    StructField('status',      StringType(),    True),
    StructField('error',       StringType(),    True),
    StructField('ingested_at', TimestampType(), True),
])
spark.createDataFrame([], ingested_files_schema).write.format('delta').mode('overwrite').saveAsTable('ingested_files')
print('ingested_files ready')
print('\nAll bootstrap tables created. Upload YAML configs to Files/configs/ next.')